<a href="https://colab.research.google.com/github/pskshyam/Full-Stack-GenAI-Bootcamp-1.0/blob/main/Class-32-18-July-2026-Data-Parsing-Extended/data_parsing_part3_docling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Input PDF
   ↓
Docling configuration
   ↓
OCR + layout analysis + table detection
   ↓
DoclingDocument
   ├── Text
   ├── Headings
   ├── Tables
   ├── Pictures
   ├── Page information
   └── Reading order
   ↓
JSON / Markdown / Text / CSV / Images

In [1]:
! pip install -q docling langchain-docling tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.

In [25]:
# Install the Tesseract engine and Python library

!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.4 MB]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.1 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,548 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,058 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Pac

In [26]:
import os
import pytesseract

TESSERACT_PATH = r"/usr/bin/tesseract"

if not os.path.exists(TESSERACT_PATH):
    raise FileNotFoundError(
        f"Tesseract executable not found at: {TESSERACT_PATH}"
    )

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 5.3.4


In [2]:
import json
from pathlib import Path

from docling.datamodel.base_models import InputFormat

In [3]:
InputFormat.PDF

<InputFormat.PDF: 'pdf'>

In [16]:
# print all supported document types
print([fmt.value for fmt in InputFormat])

['docx', 'doc', 'rtf', 'pptx', 'ppt', 'html', 'mhtml', 'image', 'pdf', 'asciidoc', 'md', 'csv', 'xlsx', 'xls', 'odt', 'ods', 'odp', 'xml_uspto', 'xml_jats', 'xml_xbrl', 'xml_doclang', 'dclx', 'mets_gbs', 'json_docling', 'audio', 'video', 'vtt', 'latex', 'email', 'epub', 'boxnote', 'iwork_pages', 'iwork_keynote', 'iwork_numbers', 'ebcdic', 'afp']


In [17]:
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TesseractCliOcrOptions,
)

Main Docling parser/converter

In [18]:
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

In [19]:
from docling_core.types.doc import (
    ImageRefMode,
    PictureItem,
    TableItem,
)

#### Output Directory Setup

Before running the parser, create dedicated directories to store output artifacts

In [20]:
PDF_PATH = Path("complex_rag_parsing_sample_with_sunny_image.pdf")
OUTPUT_DIR = Path("docling_parsed_output")
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"
PICTURE_DIR = OUTPUT_DIR / "extracted_pictures"
TABLE_DIR = OUTPUT_DIR / "extracted_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PICTURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")


#### Pipeline Configuration (`PdfPipelineOptions`)

Set feature parameters to control OCR, table recognition, and image handling

In [21]:
pipeline_options = PdfPipelineOptions() # PdfPipelineOptions controls specific feature toggles such as OCR execution and table structure analysis

In [23]:
# Enable table structure extraction
pipeline_options.do_table_structure = True

In [24]:
# Enable OCR
pipeline_options.do_ocr = True

In [27]:
pipeline_options.ocr_options = TesseractCliOcrOptions(
    tesseract_cmd=r"/usr/bin/tesseract",
    lang=["eng"],

    # False = selectable text + OCR where required
    # True = OCR complete pages
    force_full_page_ocr=False,
)

In [28]:
pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
)

| Name   |    Score |

| Sunny  |    90 |

| Anil   |    95 |

Row 1, Column 1 → Name

Row 1, Column 2 → Score

Row 2, Column 1 → Sunny

Row 2, Column 2 → 90


**DocumentConverter**: The primary class responsible for orchestrating the end-to-end parsing workflow

**InputFormat & PdfFormatOption**: Explicitly specify the input file format (e.g., PDF) and pass configuration options to the converter

In [29]:
# Preserve page and picture images
pipeline_options.images_scale = 2.0
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

In [30]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

The Docling workflow processes documents through a defined sequence of steps:$$\text{Input PDF} \longrightarrow \text{Pipeline Options Config} \longrightarrow \text{DocumentConverter} \longrightarrow \text{DoclingDocument Object} \longrightarrow \text{Multi-Format Exports}$$

**Export formats:** JSON, MD, TXT, CSV, PNG

In [31]:
print("Parsing PDF with Docling...")

conversion_result = converter.convert(PDF_PATH)

Parsing PDF with Docling...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

ERROR:docling.models.stages.ocr.tesseract_ocr_cli_model:OSD failed (doc complex_rag_parsing_sample_with_sunny_image.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7925c4e3acf0>):
 b'Estimating resolution as 220\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'


In [34]:
docling_document = conversion_result.document
print(type(docling_document))

<class 'docling_core.types.doc.document.DoclingDocument'>


In [35]:
print("Total pages:", len(docling_document.pages))
print("Total tables:", len(docling_document.tables))
print("Total pictures:", len(docling_document.pictures))

Total pages: 24
Total tables: 23
Total pictures: 10


#### Raw JSON Metadata Export

Generates exhaustive JSON representations (around 33,000 lines for complex files) containing precise bounding boxes, coordinates, and metadata

In [36]:
json_output_path = OUTPUT_DIR / "docling_document.json"

In [37]:
json_output_path.write_text(
    json.dumps(
        docling_document.export_to_dict(),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

8980063

In [38]:
print("JSON saved:", json_output_path)

JSON saved: docling_parsed_output/docling_document.json


#### Markdown (`.md`) and Text (`.txt`) Exports
Clean text files maintaining structural readability and tabular data

In [39]:
markdown_output_path = OUTPUT_DIR / "rag_ready_document.md"

In [40]:
docling_document.save_as_markdown(
    markdown_output_path,
    image_mode=ImageRefMode.REFERENCED,
)

print("Markdown saved:", markdown_output_path)

Markdown saved: docling_parsed_output/rag_ready_document.md


In [41]:
text_output_path = OUTPUT_DIR / "extracted_text.txt"

text_output_path.write_text(
    docling_document.export_to_markdown(strict_text=True),
    encoding="utf-8",
)

print("Plain text saved:", text_output_path)


Plain text saved: docling_parsed_output/extracted_text.txt


#### Table Exports (CSV, MD, PNG)

Tables are individually extracted and saved into standalone `.csv` files, Markdown format, and rendered `.png` images

In [43]:
for table_index, table in enumerate(
    docling_document.tables,
    start=1,
):
    table_df = table.export_to_dataframe(
        doc=docling_document
    )

    csv_path = TABLE_DIR / f"table_{table_index:03d}.csv"
    markdown_path = TABLE_DIR / f"table_{table_index:03d}.md"
    image_path = TABLE_DIR / f"table_{table_index:03d}.png"

    table_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
    )

    markdown_path.write_text(
        table_df.to_markdown(index=False),
        encoding="utf-8",
    )

    table_image = table.get_image(docling_document)

    if table_image is not None:
        table_image.save(image_path, format="PNG")

print("Tables saved:", TABLE_DIR)


Tables saved: docling_parsed_output/extracted_tables


#### Embedded Picture Crops & Page Image Renders
Extracts both page-level image renders and embedded individual figures into designated output folders

In [42]:
for page_number, page in docling_document.pages.items():

    if page.image is None:
        continue

    page_image_path = (
        PAGE_IMAGE_DIR
        / f"page_{page.page_no:03d}.png"
    )

    page.image.pil_image.save(
        page_image_path,
        format="PNG",
    )

print("Page images saved:", PAGE_IMAGE_DIR)

Page images saved: docling_parsed_output/page_images


In [44]:
picture_counter = 0

for element, _level in docling_document.iterate_items():

    if not isinstance(element, PictureItem):
        continue

    picture_counter += 1

    picture_path = (
        PICTURE_DIR
        / f"picture_{picture_counter:03d}.png"
    )

    picture_image = element.get_image(docling_document)

    if picture_image is not None:
        picture_image.save(
            picture_path,
            format="PNG",
        )

print("Pictures saved:", PICTURE_DIR)

Pictures saved: docling_parsed_output/extracted_pictures
